# Frozen confirmatory 8B activation extraction

This notebook is the Colab/VS Code execution surface for the frozen format-matched confirmation. It calls the repository's hash-pinned extraction implementation rather than duplicating the scientific logic in notebook cells.

Frozen protocol: `explicit_only`; mean layer 6 is the gate; last-token layer 6 is diagnostic only; 1,200 fitting items plus 256 previously unscored full-format controls; naturalistic transfer is never loaded. Run the cells in order. The two model cells are separate so a runtime interruption does not require repeating a completed model.

For a confirmatory result comparable to the development evidence, use an L4 or A100 and leave `LOAD_IN_4BIT = False`. A 4-bit T4 run is available for troubleshooting only and must not be substituted for the frozen unquantized verdict.

The notebook searches the open workspace, `/content`, and `MyDrive/lang-ea` for three inputs: `extract_confirmatory_8b.py`, the hash-pinned `probe_items.jsonl`, and `format_matched_controls.jsonl`. Keep the repository layout or place those three files directly in `MyDrive/lang-ea`. Outputs default to Drive so they survive a Colab runtime restart.

In [ ]:
# Install only the GPU extraction dependencies. Colab already supplies torch.
%pip install -q transformers accelerate bitsandbytes sentencepiece

In [ ]:
# Configuration
from pathlib import Path
import json
import subprocess
import sys
import torch

USE_GOOGLE_DRIVE = True      # Persistent transport for the VS Code Colab extension.
BATCH_SIZE = 4              # 4 is conservative on L4; 8 is usually fine on A100.
MODEL_LOAD_DTYPE = "bfloat16"
LOAD_IN_4BIT = False        # Troubleshooting only; not the frozen confirmatory verdict.
SKIP_COMPLETE = True        # Safely preserves a completed first model after a restart.

SWALLOW = "tokyotech-llm/Llama-3.1-Swallow-8B-Instruct-v0.3"
META = "unsloth/Meta-Llama-3.1-8B-Instruct"
MODELS = [SWALLOW, META]

EXPECTED_ITEMS_HASH = "97796dcf3f54918cb7b55c009f6458832fcf75d880df50252ffdf8415c6bc84e"
EXPECTED_CONTROLS_HASH = "fbd320168c38e14321e50cab1872144a3d32e171d4a56afca63dcb6f782131b9"

DRIVE_DIR = Path("/content/drive/MyDrive/lang-ea")
if USE_GOOGLE_DRIVE and not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

here = Path.cwd().resolve()
SEARCH_ROOTS = list(dict.fromkeys([here, *here.parents, Path("/content"), DRIVE_DIR]))

def find_input(*relative_names):
    for root in SEARCH_ROOTS:
        for name in relative_names:
            candidate = (root / name).resolve()
            if candidate.is_file():
                return candidate
    raise FileNotFoundError(
        f"Could not find any of {relative_names}. Put the file in {DRIVE_DIR}."
    )

EXTRACTOR = find_input("probe_check/confirmations/01_format_matched/extract_confirmatory_8b.py", "extract_confirmatory_8b.py")
ITEMS = find_input("data/processed/demo16_v2/probe_items.jsonl", "probe_items.jsonl")
CONTROLS = find_input("probe_check/format_matched_controls.jsonl", "format_matched_controls.jsonl")
WORKDIR = EXTRACTOR.parents[3] if EXTRACTOR.parent.name == "01_format_matched" else here
OUTPUT_ROOT = DRIVE_DIR / "confirmatory_8b" if USE_GOOGLE_DRIVE else WORKDIR / "probe_check" / "out" / "confirmatory_8b"
assert torch.cuda.is_available(), "Select a GPU runtime before continuing."
gpu = torch.cuda.get_device_properties(0)
gpu_gb = gpu.total_memory / 2**30
print(f"script:   {EXTRACTOR}")
print(f"items:    {ITEMS}")
print(f"controls: {CONTROLS}")
print(f"output: {OUTPUT_ROOT}")
print(f"GPU:    {gpu.name} ({gpu_gb:.1f} GiB)")
if not LOAD_IN_4BIT and gpu_gb < 20:
    raise RuntimeError(
        "The frozen unquantized run needs an L4/A100-class runtime. "
        "Do not silently switch to 4-bit for the confirmatory verdict."
    )
if LOAD_IN_4BIT:
    print("WARNING: 4-bit mode is troubleshooting-only and not the frozen verdict.")

In [ ]:
# Hash and structural preflight. This cell loads no model and no activations.
preflight = [
    sys.executable,
    str(EXTRACTOR),
    "--items", str(ITEMS),
    "--controls", str(CONTROLS),
    "--check-only",
]
subprocess.run(preflight, cwd=WORKDIR, check=True)

In [ ]:
# Resumable helpers and post-extraction integrity checks.
def model_dir(model_id):
    return OUTPUT_ROOT / model_id.split("/")[-1]

def load_cpu_tensor(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:  # Older Colab torch.
        return torch.load(path, map_location="cpu")

def verify_model(model_id):
    out = model_dir(model_id)
    config = json.loads((out / "config.json").read_text(encoding="utf-8"))
    assert config["model"] == model_id
    assert config["items_sha256"] == EXPECTED_ITEMS_HASH
    assert config["controls_sha256"] == EXPECTED_CONTROLS_HASH
    assert config["gating_cell"] == {"position": "mean", "layer": 6}
    assert config["load_in_4bit"] is LOAD_IN_4BIT
    if not LOAD_IN_4BIT:
        assert config["model_load_dtype"] == MODEL_LOAD_DTYPE
    fit_meta = (out / "meta.jsonl").read_text(encoding="utf-8").splitlines()
    ctl_meta = (out / "confirmatory_controls" / "meta.jsonl").read_text(encoding="utf-8").splitlines()
    assert len(fit_meta) == 1200 and len(ctl_meta) == 256
    assert all("cue_locus" in json.loads(line) for line in fit_meta)
    for position in ("mean", "last"):
        fit = load_cpu_tensor(out / f"acts_layer6_{position}.pt")
        ctl = load_cpu_tensor(out / "confirmatory_controls" / f"acts_layer6_{position}.pt")
        assert tuple(fit.shape) == (1200, 4096), (position, fit.shape)
        assert tuple(ctl.shape) == (256, 4096), (position, ctl.shape)
        del fit, ctl
    print(f"VERIFIED: {model_id} -> {out}")
    return out

def run_model(model_id, force=False):
    if SKIP_COMPLETE and not force:
        try:
            verify_model(model_id)
            print("Complete output already exists; skipping extraction.")
            return
        except (AssertionError, FileNotFoundError, KeyError, json.JSONDecodeError):
            pass
    command = [
        sys.executable,
        str(EXTRACTOR),
        "--items", str(ITEMS),
        "--controls", str(CONTROLS),
        "--models", model_id,
        "--out", str(OUTPUT_ROOT),
        "--batch-size", str(BATCH_SIZE),
        "--dtype", MODEL_LOAD_DTYPE,
    ]
    if LOAD_IN_4BIT:
        command.append("--load-in-4bit")
    subprocess.run(command, cwd=WORKDIR, check=True)
    verify_model(model_id)

## Model 1: Japanese-specialized Swallow 8B

Run this cell through `VERIFIED` before starting model 2. A completed output is skipped safely if the notebook is restarted.

In [ ]:
run_model(SWALLOW)

## Model 2: Meta-Llama 3.1 8B baseline

In [ ]:
run_model(META)

In [ ]:
# Final handoff check. This verifies tensors but does not train or score a probe.
for model_id in MODELS:
    verify_model(model_id)
print("\nBOTH 8B EXTRACTIONS ARE COMPLETE. Return the two output directories to the repo, then run the frozen evaluator exactly as documented in CONFIRMATORY_CONTROL_EXTRACTION_HANDOFF.md.")